# RAG-based Document QA Application with Pinecone
### Using Embedding, Vector DB(Pinecone), Prompt Templates, Output Parsers, and RunnablePassthrough

In [33]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser
import re 
import os
import streamlit as st 

## Environment Setup
Load environment variables such as API keys from a `.env` file to ensure secure configuration.

In [2]:
%load_ext dotenv
%dotenv

## Document Loading

In [3]:
doc = PyPDFLoader('Introduction_to_Data_and_Data_Science.pdf')
doc = doc.load()
doc

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-11-09T10:16:34+02:00', 'author': 'Hristina  Hristova', 'moddate': '2023-11-09T10:16:34+02:00', 'source': 'Introduction_to_Data_and_Data_Science.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Analysis vs Analytics \nAlright! So… \nLet’s discuss the not-so-obvious differences \nbetween the terms analysis and analytics. \nDue to the similarity of the words, some people \nbelieve they share the same meaning, and thus \nuse them interchangeably. Technically, this \nisn’t correct. There is, in fact, a distinct \ndifference between the two. And the reason \nfor one often being used instead of the other \nis the lack of a transparent understanding \nof both. \nSo, let’s clear this up, shall we? \nFirst, we will start with analysis. \nConsider the following… \nYou have a huge dataset containing data of \nvarious types. Instead of tackli

In [4]:
doc[0].page_content

'Analysis vs Analytics \nAlright! So… \nLet’s discuss the not-so-obvious differences \nbetween the terms analysis and analytics. \nDue to the similarity of the words, some people \nbelieve they share the same meaning, and thus \nuse them interchangeably. Technically, this \nisn’t correct. There is, in fact, a distinct \ndifference between the two. And the reason \nfor one often being used instead of the other \nis the lack of a transparent understanding \nof both. \nSo, let’s clear this up, shall we? \nFirst, we will start with analysis. \nConsider the following… \nYou have a huge dataset containing data of \nvarious types. Instead of tackling the entire \ndataset and running the risk of becoming overwhelmed, \nyou separate it into easier to digest chunks \nand study them individually and examine how \nthey relate to other parts. And that’s analysis \nin a nutshell. \nOne important thing to remember, however, \nis that you perform analyses on things that \nhave already happened in the 

## Token Reduction by Removing Newlines
Reducing tokens by remove '\n'

In [5]:
for i in range(len(doc)):
    doc[i].page_content = re.sub('\n','',doc[i].page_content)
    doc[i].page_content = ''.join(doc[i].page_content) 
    

In [6]:
doc[2].page_content

'as many more with a fantastic diagram. So, let’s move on! Programming Languages & Software Employed in Data Science - All the Tools You Need Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations. Moreover, you can reuse these programs whenever you need to execute the same action. As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related pr

## Text Chunking with Recursive Character Splitting
- chunk_size = 300
- overlap = 50


In [7]:
char_spliter = RecursiveCharacterTextSplitter(chunk_size = 300,
                                              chunk_overlap=50)


In [8]:
page_char_spliter = char_spliter.split_documents(doc)
page_char_spliter[0].page_content

'Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a'

## Embedding 

In [9]:
embedding = OpenAIEmbeddings(model= "text-embedding-ada-002")

In [10]:
vector_1 = embedding.embed_query(page_char_spliter[0].page_content)
vector_1

[0.00492157181724906,
 -0.025082914158701897,
 0.03172101825475693,
 -0.007936588488519192,
 -0.012237422168254852,
 -0.001961977919563651,
 -0.009874814189970493,
 -0.026197711005806923,
 0.0002203068434027955,
 -0.018482813611626625,
 0.003961960319429636,
 0.01662059873342514,
 -0.002625471679493785,
 -0.014036297798156738,
 0.008861362934112549,
 -1.534032344352454e-05,
 0.02992214262485504,
 0.01660792902112007,
 0.003968294244259596,
 -0.016075868159532547,
 -0.034178636968135834,
 -0.0037687711883336306,
 -0.029238063842058182,
 0.003797274548560381,
 -0.005174934398382902,
 0.013706926256418228,
 0.006555761676281691,
 -0.01864749938249588,
 -0.005741833709180355,
 0.01233876682817936,
 -0.011971390806138515,
 -0.005219273269176483,
 -0.01516376156359911,
 -0.004421180579811335,
 -0.020408371463418007,
 -0.01593651808798313,
 -0.00931741576641798,
 -0.014682373031973839,
 0.03232908993959427,
 0.0019730625208467245,
 0.005048253107815981,
 0.002441783668473363,
 -0.005982528440

In [11]:
text = [chunk.page_content for chunk in page_char_spliter]
text_embedding  = embedding.embed_documents(text)

## Vector Store Exploration and Inspection
Explored stored documents and embeddings using Chroma’s .get() method to verify data integrity and ensure correct indexing before retrieval.

In [12]:
vectorstore = Chroma.from_documents(documents= page_char_spliter,
                                    embedding=embedding,
                                    persist_directory="chroma_db")

In [13]:
vectorstore.persist()

C:\Users\paton\AppData\Local\Temp\ipykernel_21924\398866168.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [14]:
vectorstore = Chroma( persist_directory="chroma_db",
                      embedding_function=embedding
)

C:\Users\paton\AppData\Local\Temp\ipykernel_21924\3949492780.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma( persist_directory="chroma_db",


In [15]:
vectorstore.get()

{'ids': ['e0517bae-cf10-4a63-b773-4549eccf3f7e',
  '83c1292e-cb4f-4e03-a9c1-2a011c5131a3',
  '16d7e13d-d562-40e4-9f80-80e8e9590117',
  '4c0b10a5-1f9c-4f4f-9c37-95207d537501',
  '41701a21-1d75-4b97-a8e6-cd18a14c924c',
  '01685623-2645-4893-bff2-b611f65ba78b',
  '07ac4d80-8777-458c-9e8a-9e8a31959fc7',
  '435ca09f-b708-4ad7-a006-2ef5a64e7bb4',
  '80bb31c4-83a5-47fb-b9ce-76ae3b4e79fd',
  'a45ff0ac-8363-45bf-a32a-d242d2cb90fe',
  '31ee6015-e836-4db5-952b-8a8f712778d6',
  '1f105fd6-4f8c-4dd1-afc9-8288b25487c4',
  '083f21ad-faee-4f62-8010-38807f44971d',
  '91efa314-7bbe-4aa1-b303-849463a3222e',
  'c0297bb3-3aa1-491e-99dc-47793c050166',
  'd3bc9927-ba28-40d2-a5de-b3e9ea7a03dd',
  '99efa0d9-9669-4854-9d5c-a75d148ee8ad',
  'ce0d143e-b744-4068-b78a-41d086e27679',
  '3de217e2-25fa-41e3-bb03-4cfbd0051f9b',
  '195a2b77-0ce9-437f-92f5-1a8949c82109',
  '89a2ea5b-db3e-4f87-aad9-1317a38af40b',
  '743037d1-0cd2-4aa7-8348-43bd567e8dd8',
  '3186bc07-98d9-4d38-a5c0-2e33a7c80abe',
  'a880df05-3f2d-4832-b970-

In [16]:
vectorstore.get('83c1292e-cb4f-4e03-a9c1-2a011c5131a3')

{'ids': ['83c1292e-cb4f-4e03-a9c1-2a011c5131a3'],
 'embeddings': None,
 'documents': ['this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'producer': 'Microsoft® Word for Microsoft 365',
   'author': 'Hristina  Hristova',
   'page': 0,
   'page_label': '1',
   'moddate': '2023-11-09T10:16:34+02:00',
   'source': 'Introduction_to_Data_and_Data_Science.pdf',
   'creationdate': '2023-11-09T10:16:34+02:00',
   'creator': 'Microsoft® Word for Microsoft 365',
   'total_pages': 6}]}

## Vector Store (Pinecone)
This project uses Pinecone as a vector database to store and retrieve document embeddings efficiently.

- Enables semantic search for relevant context
- Scales for large datasets
- Supports low-latency retrieval for RAG pipeline

In [17]:
pc = Pinecone(api_key =  os.environ.get("PINECONE_API_KEY"), enviroment = os.environ.get("PINECONE_ENV"))

In [20]:
pc.create_index(
    name="rag-base-document-qa", 
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "rag-base-document-qa",
    "metric": "cosine",
    "host": "rag-base-document-qa-ipopocv.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}

In [21]:
vectorstore = PineconeVectorStore.from_documents(
    documents=page_char_spliter,
    embedding=embedding,
    index_name="rag-base-document-qa"
)

-------------------------------


## Retrieval 

In [22]:
question = "What promgramming languages do data scientists ues?"

In [23]:
retrival_doc = vectorstore.similarity_search(query=question, 
                                             k=5
                                            )

In [24]:
retrival_doc

[Document(id='5d89554c-19b3-4550-a7c8-6447f88bda87', metadata={'author': 'Hristina  Hristova', 'creationdate': '2023-11-09T10:16:34+02:00', 'creator': 'Microsoft® Word for Microsoft 365', 'moddate': '2023-11-09T10:16:34+02:00', 'page': 2.0, 'page_label': '3', 'producer': 'Microsoft® Word for Microsoft 365', 'source': 'Introduction_to_Data_and_Data_Science.pdf', 'total_pages': 6.0}, page_content='as many more with a fantastic diagram. So, let’s move on! Programming Languages & Software Employed in Data Science - All the Tools You Need Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers.'),
 Document(id='e15b1ceb-bd86-4c5d-8ada-fe3435778f24', metadata={'author': 'Hristina  Hristova', 'creationdate': '2023-11-09T10:16:34+02:00', 'creator': 'Microsoft® Word for Microsoft 365', 'moddate': '2023-11-09T10:16:34+02:00', 'page': 3.0, 'page_label': '4', 'producer': 'Microsoft® Word for Micros

In [25]:
retrival_doc = vectorstore.max_marginal_relevance_search(query=question,
                                                          k=3,
                                                          lambda_mult=1
                                                         )

In [26]:
retrival_doc

[Document(metadata={'author': 'Hristina  Hristova', 'creationdate': '2023-11-09T10:16:34+02:00', 'creator': 'Microsoft® Word for Microsoft 365', 'moddate': '2023-11-09T10:16:34+02:00', 'page': 2.0, 'page_label': '3', 'producer': 'Microsoft® Word for Microsoft 365', 'source': 'Introduction_to_Data_and_Data_Science.pdf', 'total_pages': 6.0}, page_content='as many more with a fantastic diagram. So, let’s move on! Programming Languages & Software Employed in Data Science - All the Tools You Need Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers.'),
 Document(metadata={'author': 'Hristina  Hristova', 'creationdate': '2023-11-09T10:16:34+02:00', 'creator': 'Microsoft® Word for Microsoft 365', 'moddate': '2023-11-09T10:16:34+02:00', 'page': 3.0, 'page_label': '4', 'producer': 'Microsoft® Word for Microsoft 365', 'source': 'Introduction_to_Data_and_Data_Science.pdf', 'total_pages': 6.0}, 

## Define Prompt Templates
Create prompt templates for retrieval based on user queries.

In [27]:
TEMPLATE = '''Answer the following question:{question}

To answer the following question use only the following context:{context}'''

prompt_template = PromptTemplate.from_template(TEMPLATE)


In [28]:
retriever = vectorstore.as_retriever(search_type="mmr", k=3)

## Initialize LLM Model
Set up the ChatOpenAI model with desired parameters such as temperature and token limits.

In [34]:
chat = ChatOpenAI(model='gpt-4',
                  max_completion_tokens=200,
                  temperature=0)

## Build and Run the Chains

In [30]:
chain = ({'question': RunnablePassthrough(),
          'context' : retriever}
         | prompt_template
         | chat
         | StrOutputParser())


In [31]:
chain.invoke(question)

'Data scientists use programming languages such as R, Python, SQL, and MATLAB.'